In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from fancyimpute import IterativeImputer
import re
import traceback
import json
import joblib
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

CONFIG = {
    "data_path": "your_data.xlsx",   # data not public, configure path by yourself
    "output_dir": "output",          # configure output dir by yourself
    "feature_cols": ['S', 'Vf', 'L/d', 'fc', 'W/C'],
    "target_col": 'log_Nf',
    "required_cols": ['Fiber volume fraction Vf (%)','S','L/d','fc (MPa)','W/C','Nf (cycles)','Fiber type','Data Quality Classification','Substrate type'],
    "extra_cols_for_ld": ['Fiber length (mm)', 'Fiber diameter (μm)'],
    "n_repeats": 5,
    "random_seeds": [42, 123, 456, 789, 999]
}

# Data cleaning: numeric extraction (regex details omitted)
def clean_numeric_column(series):
    # details omitted
    return pd.to_numeric(series, errors='coerce')

# Feature engineering: compute aspect ratio L/d
def calc_ld(row):
    # L/d = length(mm)*1000 / diameter(μm), details omitted
    return np.nan  # placeholder

# Load raw data from Excel and perform multiple imputation
def load_and_impute():
    # df = pd.read_excel(CONFIG["data_path"])  # data not public
    # IterativeImputer for missing values (details omitted)
    return df  # placeholder

# Data cleaning and feature engineering pipeline
def clean_and_engineer(df):
    # Filter required columns, clean numeric, compute L/d, filter steel fiber / quality A,B
    # Create log_Nf target; split Case A+B (mortar+concrete) and all_in_one
    # details omitted
    return df_combined, df_all_in_one  # placeholder

best_models = {}
raw_results = {'case_a': [], 'case_b': [], 'case_ab': [], 'all_in_one': []}
early_stop_record = {}
early_stop_history = {}
no_early_history = {}
all_resid_dict = {}

# Train XGBoost with Optuna hyperparameter search + early stopping
def train_and_evaluate_case(X, y, case_name, seed, is_first_repeat=False):
    # Skip if too few samples
    # Train/test split (test_size=0.2), StandardScaler normalization
    # XGBoost DMatrix conversion
    # details omitted
    # Optuna objective: search XGBoost hyperparameters via CV
    def objective(trial):
        params = {
            'max_depth': trial.suggest_int('max_depth', 3, 9),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'subsample': trial.suggest_float('subsample', 0.7, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'random_state': 42
        }
        # K-fold CV with early stopping (rounds=500, early_stop=20)
        # return mean CV RMSE
        # details omitted
        return rmse_mean  # placeholder
    # Optuna study: minimize CV RMSE
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=min(20, max(1, len(X_train_s)//2)))
    best_params = study.best_params
    # Train final model with early stopping (rounds=500, patience=20)
    # Train comparison model without early stopping (fixed 500 rounds)
    # Evaluate on test set: R2, RMSE, MAE
    # Record residuals for diagnostics
    # Store best model per case
    # details omitted
    return r2, rmse, mae, final_model, scaler  # placeholder

# Run experiments across 4 cases, repeated n_repeats times
def run_experiments(df_combined, df_all_in_one):
    # Split combined data into Case A (mortar) and Case B (concrete)
    # Cases: case_a, case_b, case_ab, all_in_one
    # For each repeat (seed from CONFIG), train and evaluate each case
    # Collect R2/RMSE/MAE per repeat, aggregate mean ± std
    # details omitted
    summary = {}  # placeholder
    return summary

# SHAP analysis and plots (2x2 bar + beeswarm + dependence)
def shap_analysis_and_plots(best_models, summary):
    # Create output folder for SHAP figures
    if not best_models:
        return
    # Select best case by R2; order cases as Case A / B / A+B / All-in-One
    # 2x2 subplots: bar plot of mean(|SHAP|) ± std per feature (n_repeats averaged)
    # Beeswarm summary plot per case
    # Dependence plot for top-2 features of best case
    # details omitted
    pass

# Bar chart of R2 and RMSE by case (mean ± std)
def plot_case_performance_summary(summary):
    # Extract R2/RMSE mean±std per case from summary
    # 1x2 subplot: bar charts (details omitted)
    # details omitted
    pass

# Line plots of R2/RMSE trend across repeats with error band
def plot_trends_with_shaded_error(raw_results, summary):
    # For each metric (R2, RMSE), plot per-case trend with std band
    # details omitted
    pass

# Scatter predicted vs actual and residual distribution per case
def plot_pred_vs_actual_and_residuals(best_models):
    # For each case: scatter y_test vs y_pred with 45-degree line
    # Residual histogram per case
    # details omitted
    pass

# Early stopping analysis: boxplot of stop rounds and RMSE compare
def plot_early_stop_analysis():
    # Boxplot of early stopping rounds per case
    # Bar compare RMSE: early stop vs fixed 500 rounds
    # details omitted
    pass

# Early stopping mechanism illustration (RMSE curve + stop span)
def plot_early_stop_mechanism():
    # 2x2 subplots: RMSE curve with 20-round no-improvement span and stop line
    # details omitted
    pass

# No early stopping curves (fixed 500 rounds)
def plot_no_early_curves():
    # 2x2 subplots: RMSE curve for fixed 500 rounds per case
    # details omitted
    pass

# Compare early stop vs no early stop curves per case
def plot_early_stop_compare_curves():
    # 2x2 subplots: both curves with stop annotation
    # details omitted
    pass

# Residual diagnostics: histogram, Q-Q, resid vs fitted, scale-location
def residual_diagnostics_compare(all_resid_dict):
    # For each case: Shapiro-Wilk, Breusch-Pagan, outlier count (log_print omitted)
    # 2x2 subplots for each diagnostic type
    # details omitted
    pass

# Influence plots per case (OLS influence)
def plot_influence_plots(all_resid_dict):
    # 2x2 subplots: statsmodels influence_plot per case
    # details omitted
    pass

if __name__ == "__main__":
    # Main pipeline: load -> clean -> experiments -> SHAP -> all plots
    # df_raw = load_and_impute()
    # df_combined, df_all = clean_and_engineer(df_raw)
    # summary = run_experiments(df_combined, df_all)
    # shap_analysis_and_plots(best_models, summary)
    # plot_case_performance_summary(summary)
    # plot_trends_with_shaded_error(raw_results, summary)
    # plot_pred_vs_actual_and_residuals(best_models)
    # plot_early_stop_analysis()
    # plot_early_stop_mechanism()
    # plot_no_early_curves()
    # plot_early_stop_compare_curves()
    # residual_diagnostics_compare(all_resid_dict)
    # plot_influence_plots(all_resid_dict)
    # details omitted
    pass